In [1]:
# ============================================================
# CELL 0: IMPORTS & SYSTEM CONFIGURATION
# ============================================================
import os
import re
import io
import sys
import csv
import time
import base64
import json
import random
import threading
import traceback
from datetime import datetime
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed
import queue

import cv2
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image
from bs4 import BeautifulSoup
from PyPDF2 import PdfReader

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    InvalidSessionIdException,
    NoSuchWindowException
)

import warnings
warnings.filterwarnings("ignore")

max_field_size = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_field_size)
        break
    except OverflowError:
        max_field_size = max_field_size // 2

print(f"CSV field size limit set to: {max_field_size:,}")


CSV field size limit set to: 2,147,483,647


In [2]:
# ============================================================
# CELL 1: CONFIGURATION & TUNABLE PARAMETERS
# ============================================================
TESSERACT_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

START_PAGE = 1
END_PAGE   = 699   # exclusive

MAX_WORKERS = 4   # Concurrent browser worker threads

DISTRICT_NAME = "Thane"
PINCODE       = ""

# --- URL-based pagination params (match the browser address bar) ---
PROJECT_STATE  = "27"      # Maharashtra state code in the site URL
DISTRICT_CODE  = "517"   # project_district value in the site URL (use "Array" for all, or a specific district code)

BASE_SEARCH_URL = "https://maharera.maharashtra.gov.in/projects-search-result"

OUTPUT_CSV          = "Thane_scrape_output.csv"
CHECKPOINT_FILE     = "checkpoint.json"
FAILED_PAGES_CSV    = "failed_pages.csv"
FAILED_PROJECTS_CSV = "failed_projects.csv"
RUN_LOG_FILE        = "run_log.txt"

EXCEL_CHAR_LIMIT    = 32000
OUTPUT_XLSX         = OUTPUT_CSV.rsplit(".", 1)[0] + ".xlsx"
AUTO_EXPORT_EXCEL   = True

STANDARD_COLUMNS = [
    "Registration Number", "Date of Registration", "Rera ID", "Project Location",
    "Project Type", "Project Name", "survey_no",
    "Permissible Builtup Area", "Sanctioned Builtup Area", "Total Land Area",
    "Aggregate area", "Land Area for Project Applied for this Registration",
    "Project proposed completion date original", "Project proposed completion date revised",
    "Longitude", "Latitude", "Are there any Investor other than the Promoter in the Project",
    "State_UT", "District", "Taluka", "Village", "Pin_Code",
    "Boundaries_East", "Boundaries_West", "Boundaries_South",
    "Boundaries_North", "Boundaries_North_part2",
    "Promoter_Details", "Promoter_Details_part2", "Promoter_Details_part3", "Promoter_Details_part4", "Promoter_Details_part5",
    "bhk_details", "bhk_details_part2", "bhk_details_part3", "bhk_details_part4", "bhk_details_part5",
    "building_details", "Summary_of_Apartments/Units",
    "past_experience_info", "Litigations related to the project ?",
    "All_Parking_Details", "Bank Name", "IFSC Code", "Bank Address", "ProjectStatus",
    "Locality", "Organization", "Last_Modified",
    "Commencing From", "Leftmost Date", "Ending With"
]

MAX_CAPTCHA_OCR_ATTEMPTS = 3
CERT_LOAD_TIMEOUT        = 12  # seconds to wait for certificate modal/PDF per attempt
CERT_MAX_RETRIES         = 3   # how many times to re-click & retry if cert fails to load
REFRESH_ATTEMPTS         = 2
MAX_PAGE_RETRIES         = 3
MAX_PROJECT_RETRIES      = 3
PAGE_LOAD_WAIT           = 3
RETRY_BACKOFF_SECONDS    = 4

REFRESH_PAGE_ON_NAV_FAIL = True
PAGE_REFRESH_ATTEMPTS    = 2
RESTART_DRIVER_ON_CONSECUTIVE_FAILURES = 2

SKIP_FAILED_PAGES    = True
SKIP_FAILED_PROJECTS = True
AUTO_RETRY_FAILED_AT_END = True

HEADLESS = False

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH


In [3]:
# ============================================================
# CELL 2: LOGGING, CHECKPOINT & CLEAN CSV ENGINE
# ============================================================
log_lock = threading.Lock()
csv_lock = threading.Lock()
checkpoint_lock = threading.Lock()

def log(msg):
    thread_name = threading.current_thread().name
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [{thread_name}] {msg}"
    with log_lock:
        try:
            print(line)
        except UnicodeEncodeError:
            print(line.encode("ascii", errors="replace").decode("ascii"))
        try:
            with open(RUN_LOG_FILE, "a", encoding="utf-8") as f:
                f.write(line + "\n")
        except Exception:
            pass


_csv_ids_cache = None
_csv_ids_cache_lock = threading.Lock()

def is_reg_start(val):
    v = str(val).strip().strip("'\"")
    return v.startswith('P') and len(v) >= 11 and v[1:12].isdigit()


def load_existing_csv_rera_ids(force_reload=False):
    global _csv_ids_cache
    with _csv_ids_cache_lock:
        if _csv_ids_cache is not None and not force_reload:
            return set(_csv_ids_cache)

        scraped_ids = set()
        if os.path.exists(OUTPUT_CSV):
            try:
                df = pd.read_csv(OUTPUT_CSV, usecols=["Registration Number"], dtype=str)
                scraped_ids = set(df["Registration Number"].dropna().astype(str).str.strip().tolist())
            except Exception:
                try:
                    df = pd.read_csv(OUTPUT_CSV, usecols=["Rera ID"], dtype=str)
                    scraped_ids = set(df["Rera ID"].dropna().astype(str).str.strip().tolist())
                except Exception:
                    try:
                        with open(OUTPUT_CSV, "r", encoding="utf-8", errors="replace") as f:
                            reader = csv.reader(f)
                            for row in reader:
                                if row and is_reg_start(row[0]):
                                    scraped_ids.add(row[0].strip())
                    except Exception as e:
                        log(f"Warning reading existing RERA IDs from CSV: {e}")

        _csv_ids_cache = scraped_ids
        return set(_csv_ids_cache)


def add_scraped_id_to_cache(rera_id):
    global _csv_ids_cache
    if not rera_id:
        return
    with _csv_ids_cache_lock:
        if _csv_ids_cache is None:
            _csv_ids_cache = set()
        _csv_ids_cache.add(str(rera_id).strip())


def load_checkpoint():
    default_data = {
        "scraped_projects": [],
        "scraped_ids": [],
        "completed_pages": [],
        "last_interrupted": None,
        "job_start_page": START_PAGE
    }

    with checkpoint_lock:
        if os.path.exists(CHECKPOINT_FILE):
            try:
                with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
                    data = json.load(f)

                completed = sorted(set(data.get("completed_pages", [])))
                data["completed_pages"] = completed
                inferred_floor = completed[0] if completed else START_PAGE
                stored_floor = data.get("job_start_page", inferred_floor)
                data["job_start_page"] = min(stored_floor, inferred_floor, START_PAGE)

                log(f"Granular checkpoint loaded ({len(data.get('scraped_projects', []))} scraped projects, "
                    f"{len(completed)} completed pages, job_start_page={data['job_start_page']}).")
                return data
            except Exception as e:
                log(f"Could not read checkpoint file ({e}), starting clean checkpoint.")

    csv_ids = load_existing_csv_rera_ids()
    if csv_ids:
        default_data["scraped_ids"] = list(csv_ids)
        log(f"Loaded {len(csv_ids)} already scraped RERA IDs from {OUTPUT_CSV}.")

    return default_data


def save_checkpoint_data(checkpoint_data):
    with checkpoint_lock:
        try:
            if "completed_pages" in checkpoint_data:
                checkpoint_data["completed_pages"] = sorted(set(checkpoint_data["completed_pages"]))
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(checkpoint_data, f, indent=2)
        except Exception as e:
            log(f"Error saving checkpoint file: {e}")


def is_project_already_scraped(checkpoint_data, project_url, rera_id=None):
    with checkpoint_lock:
        scraped_urls = set(checkpoint_data.get("scraped_projects", []))
        scraped_ids = set(checkpoint_data.get("scraped_ids", []))

    if project_url and project_url in scraped_urls:
        return True
    if rera_id and str(rera_id).strip() in scraped_ids:
        return True
    if rera_id:
        csv_ids = load_existing_csv_rera_ids()
        if str(rera_id).strip() in csv_ids:
            return True
    return False


def mark_project_completed(checkpoint_data, project_url, rera_id=None):
    with checkpoint_lock:
        if project_url and project_url not in checkpoint_data["scraped_projects"]:
            checkpoint_data["scraped_projects"].append(project_url)
        if rera_id and str(rera_id).strip() not in checkpoint_data["scraped_ids"]:
            checkpoint_data["scraped_ids"].append(str(rera_id).strip())
        if checkpoint_data.get("last_interrupted"):
            if checkpoint_data["last_interrupted"].get("project_url") == project_url:
                checkpoint_data["last_interrupted"] = None
    save_checkpoint_data(checkpoint_data)


def record_interrupted_project(checkpoint_data, page, project_index, project_url, reason="in_progress"):
    with checkpoint_lock:
        checkpoint_data["last_interrupted"] = {
            "page": page,
            "project_index": project_index,
            "project_url": project_url,
            "reason": reason,
            "updated_at": str(datetime.now())
        }
    save_checkpoint_data(checkpoint_data)


def mark_page_completed(checkpoint_data, page):
    with checkpoint_lock:
        if page not in checkpoint_data["completed_pages"]:
            checkpoint_data["completed_pages"].append(page)
            checkpoint_data["completed_pages"].sort()
    save_checkpoint_data(checkpoint_data)


def log_failed_page(page, reason):
    row = pd.DataFrame([{"page": page, "reason": reason, "time": str(datetime.now())}])
    with csv_lock:
        row.to_csv(FAILED_PAGES_CSV, index=False, mode="a",
                   header=not os.path.exists(FAILED_PAGES_CSV))


def log_failed_project(page, project_index, project_url, reason):
    row = pd.DataFrame([{"page": page, "project_index": project_index,
                          "project_url": project_url, "reason": reason, "time": str(datetime.now())}])
    with csv_lock:
        row.to_csv(FAILED_PROJECTS_CSV, index=False, mode="a",
                   header=not os.path.exists(FAILED_PROJECTS_CSV))


def get_failed_pages():
    if not os.path.exists(FAILED_PAGES_CSV):
        return []
    try:
        with csv_lock:
            df = pd.read_csv(FAILED_PAGES_CSV)
            return sorted(set(df["page"].tolist()))
    except Exception:
        return []


def sanitize_dataframe_strings(df):
    clean_df = df.copy()
    for col in clean_df.columns:
        if clean_df[col].dtype == object or clean_df[col].dtype == 'string':
            def _clean_cell(v):
                if v is None or pd.isna(v):
                    return ""
                s = str(v).strip()
                if s == "{}" or (s.startswith("{") and s.endswith("}")):
                    return s
                return " ".join(s.split())
            clean_df[col] = clean_df[col].apply(_clean_cell)
    return clean_df


def split_all_large_columns(df, limit=EXCEL_CHAR_LIMIT):
    df_copy = sanitize_dataframe_strings(df)
    cols_to_check = list(df_copy.columns)

    for col in cols_to_check:
        s_series = df_copy[col].astype(str)
        max_len = s_series.str.len().max()
        if max_len <= limit:
            continue

        n_parts = int(max_len // limit) + 1
        part_cols = {f"{col}_part{part+2}": [] for part in range(n_parts - 1)}

        for val in s_series:
            chunks = [val[i:i+limit] for i in range(0, len(val), limit)]
            for part in range(n_parts - 1):
                part_cols[f"{col}_part{part+2}"].append(
                    chunks[part+1] if part+1 < len(chunks) else ""
                )

        df_copy[col] = s_series.str[:limit]
        col_idx = df_copy.columns.get_loc(col)
        for i, (part_name, part_data) in enumerate(part_cols.items()):
            df_copy.insert(col_idx + 1 + i, part_name, part_data)

    return df_copy


def save_scraped_project_row(df, output_csv_path, limit=EXCEL_CHAR_LIMIT):
    split_df = split_all_large_columns(df, limit=limit)

    if not os.path.exists(output_csv_path):
        initial_cols = list(STANDARD_COLUMNS)
        for col in split_df.columns:
            if col not in initial_cols:
                base_col = col.split("_part")[0]
                insert_idx = len(initial_cols)
                for idx, c in enumerate(initial_cols):
                    if c == base_col or c.startswith(f"{base_col}_part"):
                        insert_idx = idx + 1
                initial_cols.insert(insert_idx, col)

        row_to_write = split_df.reindex(columns=initial_cols, fill_value="")
        row_to_write.to_csv(output_csv_path, index=False, encoding="utf-8")
        return split_df

    try:
        existing_cols = pd.read_csv(output_csv_path, nrows=0).columns.tolist()
    except Exception:
        existing_cols = []

    if not existing_cols:
        split_df.to_csv(output_csv_path, index=False, encoding="utf-8")
        return split_df

    new_cols = [c for c in split_df.columns if c not in existing_cols]
    if not new_cols:
        row_to_append = split_df.reindex(columns=existing_cols, fill_value="")
        row_to_append.to_csv(output_csv_path, mode="a", index=False, header=False, encoding="utf-8")
    else:
        log(f"Dynamic schema expansion: adding new columns {new_cols} to {output_csv_path}...")
        try:
            full_existing_df = pd.read_csv(output_csv_path, dtype=str)
            merged_cols = list(existing_cols)
            for nc in new_cols:
                base_col = nc.split("_part")[0]
                insert_idx = len(merged_cols)
                for idx, c in enumerate(merged_cols):
                    if c == base_col or c.startswith(f"{base_col}_part"):
                        insert_idx = idx + 1
                merged_cols.insert(insert_idx, nc)

            full_existing_df = full_existing_df.reindex(columns=merged_cols, fill_value="")
            row_to_append = split_df.reindex(columns=merged_cols, fill_value="")
            combined = pd.concat([full_existing_df, row_to_append], ignore_index=True)
            combined.to_csv(output_csv_path, index=False, encoding="utf-8")
        except Exception as e:
            log(f"Error during schema expansion ({e}), appending aligned row directly.")
            row_to_append = split_df.reindex(columns=existing_cols, fill_value="")
            row_to_append.to_csv(output_csv_path, mode="a", index=False, header=False, encoding="utf-8")

    return split_df


def export_and_verify_dataset(csv_path=OUTPUT_CSV, xlsx_path=OUTPUT_XLSX, limit=EXCEL_CHAR_LIMIT):
    if not os.path.exists(csv_path):
        log(f"No CSV file found at {csv_path} to export.")
        return False

    try:
        log(f"Loading {csv_path} for Excel export and character validation...")
        df = pd.read_csv(csv_path, dtype=str)
        df_split = split_all_large_columns(df, limit=limit)

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df_split.to_excel(writer, index=False, sheet_name="RERA_Projects")

        log(f"✅ Excel export verified and written successfully to {xlsx_path} ({len(df_split)} rows, {len(df_split.columns)} columns).")
        return True
    except Exception as e:
        log(f"⚠️ Excel export encountered an issue: {e}")
        return False


In [4]:
# ============================================================
# CELL 3: ADAPTIVE DRIVER MANAGEMENT, DISTRICT FILTER & PAGINATION
# ============================================================
thread_local_storage = threading.local()

def create_driver(headless=True):
    options = Options()
    if headless:
       options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.page_load_strategy = "normal"
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(45)
    return driver


def driver_is_alive(driver):
    if driver is None:
        return False
    try:
        driver.execute_script("return 1")
        return True
    except Exception:
        return False


def get_thread_driver():
    if not hasattr(thread_local_storage, "driver") or thread_local_storage.driver is None:
        log("🌐 Initializing dedicated Chrome WebDriver for worker thread...")
        thread_local_storage.driver = create_driver(HEADLESS)
    else:
        if not driver_is_alive(thread_local_storage.driver):
            log("💥 Thread driver was disconnected or died — restarting Chrome...")
            try:
                thread_local_storage.driver.quit()
            except Exception:
                pass
            thread_local_storage.driver = create_driver(HEADLESS)
    return thread_local_storage.driver


def close_thread_driver():
    if hasattr(thread_local_storage, "driver") and thread_local_storage.driver is not None:
        try:
            thread_local_storage.driver.quit()
        except Exception:
            pass
        thread_local_storage.driver = None


def safe_get(driver, url, refresh_attempts=REFRESH_ATTEMPTS):
    for attempt in range(1, refresh_attempts + 2):
        try:
            if not driver_is_alive(driver):
                return False
            driver.get(url)
            time.sleep(PAGE_LOAD_WAIT)
            ready_state = driver.execute_script("return document.readyState")
            if ready_state == "complete":
                return True
        except (TimeoutException, WebDriverException) as e:
            log(f"⚠️ Navigation issue (attempt {attempt}): {e}")
        wait_s = RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 2)
        log(f"🔄 Retrying page load in {wait_s:.1f}s (attempt {attempt}/{refresh_attempts + 1})...")
        time.sleep(wait_s)
        try:
            if driver_is_alive(driver):
                driver.refresh()
                time.sleep(PAGE_LOAD_WAIT)
        except Exception:
            pass
    return False


_driver_current_page = {}
_driver_page_state_lock = threading.Lock()

def _driver_key(driver):
    try:
        return driver.session_id
    except Exception:
        return id(driver)


def reset_driver_page_state(driver=None):
    with _driver_page_state_lock:
        if driver is not None:
            _driver_current_page.pop(_driver_key(driver), None)
        else:
            _driver_current_page.clear()


def select_district_dropdown(driver, district_name, max_retries=3):
    if not district_name:
        return True

    target_name = district_name.strip()
    xpath_native = (
        "//select[contains(translate(@formcontrolname,'DISTRICT','district'),'district') "
        "or contains(translate(@id,'DISTRICT','district'),'district') "
        "or contains(translate(@name,'DISTRICT','district'),'district')]"
    )

    for attempt in range(1, max_retries + 1):
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, xpath_native)))
            dropdown_el = driver.find_element(By.XPATH, xpath_native)

            def check_options(d):
                try:
                    el = d.find_element(By.XPATH, xpath_native)
                    s = Select(el)
                    opts = [o.text.strip() for o in s.options if o.text and o.text.strip()]
                    return len(opts) > 3 or any(target_name.lower() in o.lower() for o in opts)
                except Exception:
                    return False

            try:
                WebDriverWait(driver, 20, poll_frequency=1).until(check_options)
            except TimeoutException:
                pass

            select_obj = Select(dropdown_el)
            matched_option = None
            matched_value = None
            for opt in select_obj.options:
                opt_txt = opt.text.strip()
                if opt_txt.lower() == target_name.lower() or target_name.lower() in opt_txt.lower():
                    matched_option = opt_txt
                    matched_value = opt.get_attribute("value")
                    break

            if matched_option:
                try:
                    select_obj.select_by_visible_text(matched_option)
                except Exception:
                    pass

                driver.execute_script("""
                    var sel = arguments[0];
                    var val = arguments[1];
                    if (val) sel.value = val;
                    sel.dispatchEvent(new Event('change', { bubbles: true }));
                    sel.dispatchEvent(new Event('input', { bubbles: true }));
                """, dropdown_el, matched_value)

                log(f"📍 District set to '{matched_option}' (native <select>).")
                time.sleep(1)
                return True
            else:
                current_opts = [o.text.strip() for o in select_obj.options if o.text.strip()]
                log(f"⚠️ District '{target_name}' not yet in dropdown options ({current_opts[:4]}...). Waiting...")

        except Exception as e:
            log(f"⚠️ District selection attempt {attempt}/{max_retries} warning: {e}")

        try:
            custom_control = driver.find_elements(By.XPATH, "(//ng-select | //div[contains(@class,'ng-select')])[1]")
            if custom_control and custom_control[0].is_displayed():
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", custom_control[0])
                custom_control[0].click()
                time.sleep(0.5)
                opt = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((
                    By.XPATH, f"//*[contains(translate(text(),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), '{target_name.lower()}')]"
                )))
                opt.click()
                log(f"📍 District set to '{target_name}' (custom dropdown).")
                return True
        except Exception:
            pass

        if attempt < max_retries:
            time.sleep(3)
            try:
                if driver_is_alive(driver):
                    driver.refresh()
                    time.sleep(PAGE_LOAD_WAIT)
            except Exception:
                pass

    log(f"❌ Could not select district '{district_name}' after {max_retries} attempts.")
    return False


def fill_pincode(driver, pincode):
    if not pincode:
        return True
    try:
        pin_input = driver.find_element(
            By.XPATH,
            "//input[contains(@placeholder,'Pincode') or contains(@placeholder,'PIN') "
            "or contains(translate(@formcontrolname,'PINCODE','pincode'),'pincode')]"
        )
        pin_input.clear()
        pin_input.send_keys(str(pincode))
        log(f"📮 Pincode set to '{pincode}'.")
        return True
    except Exception:
        return False


def click_search_button(driver):
    candidate_xpaths = [
        "//button[normalize-space()='Search']",
        "//button[contains(translate(normalize-space(.),'SEARCH','search'),'search')]",
        "//input[@type='submit' and contains(translate(@value,'SEARCH','search'),'search')]",
        "//a[normalize-space()='Search']",
        "//*[self::button or self::a][contains(translate(normalize-space(.),'SEARCH','search'),'search') and not(contains(translate(normalize-space(.),'RESET','reset'),'reset'))]"
    ]

    for xpath in candidate_xpaths:
        try:
            elements = [el for el in driver.find_elements(By.XPATH, xpath) if el.is_displayed()]
            for el in elements:
                try:
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                    time.sleep(0.2)
                    driver.execute_script("arguments[0].click();", el)
                    return True
                except Exception:
                    continue
        except Exception:
            continue
    return False


def check_and_dismiss_no_records_popup(driver):
    try:
        indicators = [
            "//*[contains(translate(normalize-space(.), 'NO RECORDS FOUND', 'no records found'), 'no records found')]",
            "//div[contains(@class, 'modal') and contains(@class, 'show') and contains(., 'No Records Found')]",
            "//div[contains(@class, 'modal') and (contains(., 'Search Record') or contains(., 'No Records Found'))]"
        ]
        popup_found = False
        for xpath in indicators:
            try:
                elements = [el for el in driver.find_elements(By.XPATH, xpath) if el.is_displayed()]
                if elements:
                    popup_found = True
                    break
            except Exception:
                continue

        if popup_found:
            log("⚠️ 'Search Record: No Records Found' popup detected! Dismissing and recovering...")
            close_button_xpaths = [
                "//div[contains(@class,'modal') and contains(@class,'show')]//button[normalize-space()='Close' or contains(translate(.,'CLOSE','close'),'close')]",
                "//button[normalize-space()='Close']",
                "//button[contains(translate(normalize-space(.),'CLOSE','close'),'close')]",
                "//div[contains(@class,'modal')]//button[@data-bs-dismiss='modal']",
                "//div[contains(@class,'modal')]//button[contains(@class,'btn-close')]"
            ]
            for btn_xp in close_button_xpaths:
                try:
                    btns = [b for b in driver.find_elements(By.XPATH, btn_xp) if b.is_displayed()]
                    for b in btns:
                        driver.execute_script("arguments[0].click();", b)
                        time.sleep(0.5)
                        break
                except Exception:
                    pass

            try:
                ActionChains(driver).send_keys(Keys.ESCAPE).perform()
                driver.execute_script("""
                    document.querySelectorAll('div.modal.show').forEach(function(m){
                        m.classList.remove('show'); m.style.display = 'none';
                    });
                    document.querySelectorAll('.modal-backdrop').forEach(function(b){ b.remove(); });
                    document.body.classList.remove('modal-open');
                """)
            except Exception:
                pass
            time.sleep(1)
            return True
    except Exception:
        pass
    return False


def apply_search_filters_and_search(driver, max_retries=3):
    for attempt in range(1, max_retries + 1):
        if not safe_get(driver, BASE_SEARCH_URL, REFRESH_ATTEMPTS):
            log(f"⚠️ safe_get failed to reach search page (attempt {attempt}/{max_retries})")
            continue

        if not select_district_dropdown(driver, DISTRICT_NAME):
            log(f"⚠️ Failed to select district '{DISTRICT_NAME}' on attempt {attempt}/{max_retries}.")
            continue

        fill_pincode(driver, PINCODE)

        if not click_search_button(driver):
            log(f"❌ Could not click Search button (attempt {attempt}/{max_retries})")
            continue

        # Wait up to 15s for results / pagination to populate via AJAX
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class,'row') and contains(@class,'shadow')] | "
                    "//pagination-controls | //ul[contains(@class,'pagination')] | "
                    "//div[contains(@class,'modal') and contains(@class,'show')]"
                ))
            )
        except TimeoutException:
            pass

        if check_and_dismiss_no_records_popup(driver):
            log(f"🔄 'No Records Found' after search (attempt {attempt}/{max_retries}). Retrying search...")
            continue

        with _driver_page_state_lock:
            _driver_current_page[_driver_key(driver)] = 1
        return True

    log("❌ apply_search_filters_and_search failed after all retries.")
    return False


def click_pagination_target(driver, next_page_number):
    # 1. Wait for paginator or results to be present
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.XPATH,
                "//pagination-controls | //ul[contains(@class,'pagination')] | "
                "//*[contains(@class,'page-link')] | //a[contains(.,'Next')] | //button[contains(.,'Next')]"
            ))
        )
    except TimeoutException:
        pass

    # 2. Scroll to bottom so pagination controls are rendered in viewport
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)
    except Exception:
        pass

    candidate_xpaths = [
        # Exact page number link/button
        f"//a[normalize-space()='{next_page_number}']",
        f"//button[normalize-space()='{next_page_number}']",
        f"//span[normalize-space()='{next_page_number}']/parent::a",
        f"//*[contains(@class,'page-link') and normalize-space()='{next_page_number}']",
        # Next buttons
        "//li[contains(translate(@class,'NEXT','next'),'next') and not(contains(translate(@class,'DISABLED','disabled'),'disabled'))]//a",
        "//a[normalize-space()='Next' or @aria-label='Next' or @aria-label='next' or contains(., '»')]",
        "//button[normalize-space()='Next' or @aria-label='Next' or @aria-label='next' or contains(., '»')]",
        "//a[contains(@class,'page-link') and contains(translate(.,'NEXT','next'),'next')]",
        "//pagination-controls//li[not(contains(@class,'disabled'))]//a[contains(.,'Next') or contains(., '»')]"
    ]

    for xpath in candidate_xpaths:
        try:
            elements = driver.find_elements(By.XPATH, xpath)
            for el in elements:
                if not el.is_displayed():
                    continue
                try:
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                    time.sleep(0.3)
                    driver.execute_script("arguments[0].click();", el)
                    
                    # Wait up to 12s for results on next page to load
                    try:
                        WebDriverWait(driver, 12).until(
                            EC.presence_of_element_located((By.XPATH, "//div[contains(@class,'row') and contains(@class,'shadow')]"))
                        )
                    except TimeoutException:
                        pass
                    time.sleep(1)
                    return True
                except Exception:
                    continue
        except Exception:
            continue
    return False


def select_go_to_page_dropdown(driver, target_page):
    try:
        dropdown = WebDriverWait(driver, 5).until(EC.presence_of_element_located((
            By.XPATH,
            "//*[contains(translate(normalize-space(text()),'GOTOPAGE','gotopage'),'go to page')]"
            "/following::select[1] | //select[contains(translate(@id,'PAGE','page'),'page') "
            "or contains(translate(@name,'PAGE','page'),'page') "
            "or contains(translate(@formcontrolname,'PAGE','page'),'page')]"
        )))
    except TimeoutException:
        return None

    try:
        select_el = Select(dropdown)
        available_pages = []
        for opt in select_el.options:
            txt = (opt.text or "").strip()
            if txt.isdigit():
                available_pages.append(int(txt))

        if not available_pages:
            return None

        not_over = [p for p in available_pages if p <= target_page]
        chosen_page = max(not_over) if not_over else min(available_pages)

        with _driver_page_state_lock:
            cur = _driver_current_page.get(_driver_key(driver), 1)
        if chosen_page == cur:
            return chosen_page

        select_el.select_by_visible_text(str(chosen_page))
        time.sleep(PAGE_LOAD_WAIT)
        log(f"🎯 Jumped to page {chosen_page} via 'Go to page' dropdown.")
        return chosen_page
    except Exception as e:
        log(f"⚠️ 'Go to page' dropdown jump skipped ({e}).")
        return None




def _build_page_url(page_number):
    """Build the direct search-result URL for *page_number* using configured URL params."""
    return (
        f"{BASE_SEARCH_URL}"
        f"?project_name=&project_location=&project_completion_date="
        f"&project_state={PROJECT_STATE}"
        f"&project_district={DISTRICT_CODE}"
        f"&carpetAreas=&completionPercentages=&project_division="
        f"&page={page_number}&op=Search"
    )


def go_to_results_page(driver, target_page, refresh_on_fail=True):
    """Navigate directly to *target_page* via URL instead of clicking UI elements."""
    max_nav_attempts = 2 if refresh_on_fail else 1

    for attempt in range(1, max_nav_attempts + 1):
        page_url = _build_page_url(target_page)
        log(f"Link Navigating directly to page {target_page} via URL (attempt {attempt}/{max_nav_attempts}): {page_url}")

        if not safe_get(driver, page_url, REFRESH_ATTEMPTS):
            log(f"Warning safe_get failed for page {target_page} URL (attempt {attempt}/{max_nav_attempts})")
            time.sleep(RETRY_BACKOFF_SECONDS)
            continue

        # Wait for project result cards or pagination to confirm page loaded
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//div[contains(@class,'row') and contains(@class,'shadow')] | "
                    "//pagination-controls | //ul[contains(@class,'pagination')] | "
                    "//div[contains(@class,'modal') and contains(@class,'show')]"
                ))
            )
        except TimeoutException:
            pass

        # Dismiss any 'No Records Found' popup
        if check_and_dismiss_no_records_popup(driver):
            log(f"Retry No Records Found on page {target_page} URL. Retrying (attempt {attempt}/{max_nav_attempts})...")
            time.sleep(RETRY_BACKOFF_SECONDS)
            continue

        # Success - update tracked page state
        with _driver_page_state_lock:
            _driver_current_page[_driver_key(driver)] = target_page
        log(f"Success Successfully landed on page {target_page} via direct URL.")
        return True

    reset_driver_page_state(driver)
    log(f"Failed go_to_results_page: failed to reach page {target_page} after {max_nav_attempts} attempts.")
    return False


In [5]:
# ============================================================
# CELL 4: FAST CERTIFICATE EXTRACTION & OPENCV CAPTCHA SOLVER
# ============================================================
def close_modal(driver, modal_locator=(By.CSS_SELECTOR, "div.modal.show"), timeout=4):
    try:
        modal = WebDriverWait(driver, 2).until(EC.visibility_of_element_located(modal_locator))
    except TimeoutException:
        return

    try:
        close_btn = modal.find_element(By.CSS_SELECTOR, "button.btn-close, [data-bs-dismiss='modal']")
        driver.execute_script("arguments[0].click();", close_btn)
        WebDriverWait(driver, timeout).until(EC.invisibility_of_element_located(modal_locator))
        return
    except Exception:
        pass

    try:
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    except Exception:
        pass

    try:
        driver.execute_script("""
            document.querySelectorAll('div.modal.show').forEach(function(m){
                m.classList.remove('show'); m.style.display = 'none';
            });
            document.querySelectorAll('.modal-backdrop').forEach(function(b){ b.remove(); });
            document.body.classList.remove('modal-open');
        """)
    except Exception:
        pass



def extract_certificate_dates(driver, button, wait):
    """Click a certificate button, wait for the PDF modal with a per-attempt timer,
    and retry (re-click) up to CERT_MAX_RETRIES times if it fails to load.
    """
    all_dates = []

    for attempt in range(1, CERT_MAX_RETRIES + 1):
        modal_opened = False
        try:
            # ── close any leftover modal before (re)clicking ──────────────────
            close_modal(driver)
            time.sleep(0.3)

            # ── scroll into view and click the certificate button ─────────────
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
            time.sleep(0.2)
            driver.execute_script("arguments[0].click();", button)

            # ── wait for modal visibility with CERT_LOAD_TIMEOUT ──────────────
            try:
                WebDriverWait(driver, CERT_LOAD_TIMEOUT).until(
                    EC.visibility_of_element_located((By.ID, "exampleModal"))
                )
                modal_opened = True
            except TimeoutException:
                log(f"⏱️ Certificate modal did not appear within {CERT_LOAD_TIMEOUT}s "
                    f"(attempt {attempt}/{CERT_MAX_RETRIES}). Retrying...")
                close_modal(driver)
                time.sleep(2)
                continue  # retry the click

            # ── wait for pdfHolder and <object> with CERT_LOAD_TIMEOUT ─────────
            try:
                pdf_holder = WebDriverWait(driver, CERT_LOAD_TIMEOUT).until(
                    EC.presence_of_element_located((By.CLASS_NAME, "pdfHolder"))
                )
            except TimeoutException:
                log(f"⏱️ Certificate pdfHolder did not appear within {CERT_LOAD_TIMEOUT}s "
                    f"(attempt {attempt}/{CERT_MAX_RETRIES}). Retrying...")
                close_modal(driver, (By.ID, "exampleModal"))
                time.sleep(2)
                continue  # retry the click

            obj_tag = pdf_holder.find_element(By.TAG_NAME, "object")

            # ── poll for the base64 PDF data to be populated ──────────────────
            pdf_data = None
            try:
                WebDriverWait(driver, CERT_LOAD_TIMEOUT).until(
                    lambda d: obj_tag.get_attribute("data") and
                              obj_tag.get_attribute("data").startswith("data:application/pdf;base64,")
                )
                pdf_data = obj_tag.get_attribute("data")
            except TimeoutException:
                log(f"⏱️ Certificate PDF data did not populate within {CERT_LOAD_TIMEOUT}s "
                    f"(attempt {attempt}/{CERT_MAX_RETRIES}). Retrying...")
                close_modal(driver, (By.ID, "exampleModal"))
                time.sleep(2)
                continue  # retry the click

            # ── PDF loaded — extract dates ─────────────────────────────────────
            if pdf_data and pdf_data.startswith("data:application/pdf;base64,"):
                pdf_bytes = base64.b64decode(pdf_data.split(",")[1])
                pdf_file = io.BytesIO(pdf_bytes)
                reader = PdfReader(pdf_file)

                full_text = ""
                for page in reader.pages:
                    text = page.extract_text()
                    if text:
                        full_text += text + "\n"

                date_patterns = [
                    r"\b\d{2}/\d{2}/\d{4}\b",
                    r"\b\d{2}-\d{2}-\d{4}\b",
                    r"\b\d{4}-\d{2}-\d{2}\b",
                ]
                for pattern in date_patterns:
                    all_dates.extend(re.findall(pattern, full_text))

            # ── success — close modal and stop retrying ───────────────────────
            close_modal(driver, (By.ID, "exampleModal"))
            break

        except Exception as e:
            log(f"Warning Certificate extraction attempt {attempt}/{CERT_MAX_RETRIES} failed: {type(e).__name__}: {e}")
            try:
                if modal_opened:
                    close_modal(driver, (By.ID, "exampleModal"))
                else:
                    close_modal(driver)
            except Exception:
                pass
            time.sleep(2)
            # continue to next attempt

    return {
        "Commencing From": all_dates[0] if len(all_dates) > 0 else None,
        "Leftmost Date":   all_dates[1] if len(all_dates) > 1 else None,
        "Ending With":     all_dates[-1] if len(all_dates) > 2 else None,
    }


def wait_for_element(driver, by, identifier, timeout=15):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((by, identifier)))


def wait_for_element_to_be_visible(driver, by, identifier, timeout=15):
    return WebDriverWait(driver, timeout).until(EC.visibility_of_element_located((by, identifier)))


def _get_captcha_data_url(driver, elem):
    return driver.execute_script('''
        var elem = arguments[0];
        if (!elem) return null;
        var tag = elem.tagName.toLowerCase();

        if (tag === 'img') {
            if (elem.src && elem.src.indexOf('data:image') === 0) {
                return elem.src;
            }
        }

        if (tag === 'canvas') {
            try { return elem.toDataURL('image/png'); } catch (e) { return null; }
        }

        var canvas = document.createElement('canvas');
        canvas.width = elem.naturalWidth || elem.width || elem.clientWidth || 300;
        canvas.height = elem.naturalHeight || elem.height || elem.clientHeight || 80;
        var ctx = canvas.getContext('2d');
        ctx.drawImage(elem, 0, 0, canvas.width, canvas.height);
        try { return canvas.toDataURL('image/png'); } catch (e) { return null; }
    ''', elem)


def wait_for_captcha_ready(driver, captcha_element, timeout=6):
    try:
        WebDriverWait(driver, timeout).until(lambda d: d.execute_script('''
            var el = arguments[0];
            if (!el) return false;
            if (el.tagName.toLowerCase() === 'img') {
                return el.complete && el.naturalWidth > 0;
            }
            return true;
        ''', captcha_element))
    except TimeoutException:
        pass


def save_captcha_image_from_canvas(driver, captcha_element, save_path="captcha_image.png"):
    try:
        wait_for_captcha_ready(driver, captcha_element)
        data_url = _get_captcha_data_url(driver, captcha_element)
        if data_url:
            base64_data = data_url.split(',')[1]
            img_bytes = base64.b64decode(base64_data)
            with open(save_path, "wb") as f:
                f.write(img_bytes)
            return save_path
    except Exception:
        pass

    captcha_element.screenshot(save_path)
    return save_path


def clean_captcha(captcha_text):
    captcha_text = captcha_text.upper()
    return re.sub(r"[^A-Z0-9]", "", captcha_text)


def process_captcha_with_opencv(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return ""

    img_padded = cv2.copyMakeBorder(img, 15, 15, 15, 15, cv2.BORDER_CONSTANT, value=[255, 255, 255])
    gray = cv2.cvtColor(img_padded, cv2.COLOR_BGR2GRAY)

    config = "--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    resized = cv2.resize(gray, None, fx=3.5, fy=3.5, interpolation=cv2.INTER_CUBIC)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(resized)
    text1 = pytesseract.image_to_string(enhanced, config=config).strip()
    c1 = clean_captcha(text1)
    if len(c1) == 6:
        return c1

    _, thresh_otsu = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    text2 = pytesseract.image_to_string(thresh_otsu, config=config).strip()
    c2 = clean_captcha(text2)
    if len(c2) == 6:
        return c2

    adaptive_thresh = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    text3 = pytesseract.image_to_string(adaptive_thresh, config=config).strip()
    c3 = clean_captcha(text3)
    if len(c3) == 6:
        return c3

    text4 = pytesseract.image_to_string(resized, config=config).strip()
    c4 = clean_captcha(text4)
    if len(c4) == 6:
        return c4

    candidates = [c for c in [c1, c2, c3, c4] if c]
    if candidates:
        candidates.sort(key=lambda x: (abs(len(x) - 6), -len(x)))
        return candidates[0]
    return ""


def solve_one_captcha(driver, image_name="captcha_image.png"):
    try:
        captcha_image_element = wait_for_element(driver, By.ID, "captcahCanvas", timeout=10)
        save_captcha_image_from_canvas(driver, captcha_image_element, image_name)
    except StaleElementReferenceException:
        time.sleep(1)
        captcha_image_element = wait_for_element(driver, By.ID, "captcahCanvas", timeout=10)
        save_captcha_image_from_canvas(driver, captcha_image_element, image_name)

    raw_text = process_captcha_with_opencv(image_name)
    return clean_captcha(raw_text)


def submit_captcha(driver, captcha_text):
    captcha_input = wait_for_element(driver, By.XPATH, "//input[@type='text']", timeout=10)
    captcha_input.clear()
    captcha_input.send_keys(captcha_text)
    submit_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[@class='btn btn-primary next']"))
    )
    driver.execute_script("arguments[0].click();", submit_button)
    time.sleep(2)


def captcha_was_rejected(driver):
    try:
        wait_for_element(driver, By.XPATH,
                          "//p[@id='letter'][contains(text(), 'Invalid Captcha')]", timeout=5)
        return True
    except TimeoutException:
        return False


def dismiss_invalid_captcha_popup(driver):
    try:
        wait_for_element(driver, By.XPATH, "//button[contains(@class, 'btn-primary-messagebox')]", timeout=5).click()
        time.sleep(1.5)
    except Exception:
        pass


def solve_captcha_fully_automated(driver, image_name="captcha_image.png"):
    for refresh_round in range(REFRESH_ATTEMPTS + 1):
        for ocr_attempt in range(1, MAX_CAPTCHA_OCR_ATTEMPTS + 1):
            try:
                captcha_text = solve_one_captcha(driver, image_name)
                log(f"🔎 Captcha OCR attempt {ocr_attempt} -> '{captcha_text}'")
                submit_captcha(driver, captcha_text)

                if captcha_was_rejected(driver):
                    log("❌ Invalid captcha, retrying automatically...")
                    dismiss_invalid_captcha_popup(driver)
                    continue
                else:
                    log("✅ Captcha accepted.")
                    return True
            except Exception as e:
                log(f"⚠️ Captcha attempt error: {e}")
                time.sleep(1.5)

        if refresh_round < REFRESH_ATTEMPTS:
            log(f"🔄 Refreshing captcha (refresh round {refresh_round + 1}/{REFRESH_ATTEMPTS})...")
            try:
                driver.refresh()
                time.sleep(PAGE_LOAD_WAIT)
            except Exception:
                pass

    log("🚫 Captcha could not be solved automatically after all retries.")
    return False


In [6]:
# ============================================================
# CELL 5: PROJECT DETAIL EXTRACTION & TEXT SANITIZATION
# ============================================================
def clean_str(val):
    if val is None or pd.isna(val):
        return None
    s = str(val).strip()
    return " ".join(s.split()) if s else None


def safe_text(driver, xpath, by=By.XPATH, timeout=8, default=None, field_name="", required=False):
    try:
        el = WebDriverWait(driver, timeout).until(EC.visibility_of_element_located((by, xpath)))
        return clean_str(el.text)
    except (TimeoutException, NoSuchElementException, StaleElementReferenceException) as e:
        if required:
            raise
        return default


def extract_project_data(driver):
    wait = WebDriverWait(driver, 10)

    Project_Status = safe_text(driver, "//*[contains(text(), 'Project Status')]/following::div[1]",
                                field_name="Project Status")

    Registration_Number = safe_text(driver, "//*[contains(text(), 'Registration Number')]/following-sibling::*[1]",
                                     field_name="Registration Number", required=True)
    Date_of_Registration = safe_text(driver, "//*[contains(text(), 'Date of Registration')]/following-sibling::*[1]",
                                      field_name="Date of Registration")
    project_location = safe_text(driver, "//*[contains(text(), 'Project Location')]/following-sibling::*[1]",
                                  field_name="Project Location")
    project_type = safe_text(driver, "//*[contains(text(), 'Project Type')]/following-sibling::*[1]",
                              field_name="Project Type")
    project_name = safe_text(driver, "//*[contains(text(), 'Project Name')]/following-sibling::*[1]",
                              field_name="Project Name")
    project_proposed_completion_date_original = safe_text(
        driver, "//*[contains(text(), 'Proposed Completion Date (Original)')]/following-sibling::*[1]",
        field_name="Proposed Completion Date (Original)")
    project_proposed_completion_date_revised = safe_text(
        driver, "//*[contains(text(), 'Proposed Completion Date (Revised)')]/following-sibling::*[1]",
        field_name="Proposed Completion Date (Revised)")

    survey_no = safe_text(driver, "//*[contains(text(), 'Survey Number')]/following-sibling::*[1]",
                           field_name="Survey Number")
    Total_Land_Area = safe_text(driver, "//*[contains(text(), 'Total Land Area of Approved Layout')]/following-sibling::*[1]",
                                 field_name="Total Land Area")
    Land_Area_for_Project_Applied_for_this_Registration = safe_text(
        driver, "//*[contains(text(), 'Land Area for Project Applied for this Registration')]/following-sibling::*[1]",
        field_name="Land Area for Project Applied for this Registration")
    Aggregate_area_in_sqmt_of_recreational_open_space = safe_text(
        driver, "//*[contains(text(), 'Land Area for Project Applied for this Registration')]/following-sibling::*[1]",
        field_name="Aggregate area")
    Permissible_Builtup = safe_text(driver, "//*[contains(text(), 'Permissible Built-up Area')]/following-sibling::*[1]",
                                     field_name="Permissible Built-up Area")
    Sanctioned_Builtup = safe_text(driver, "//*[contains(text(), 'Sanctioned Built-up Area of the Project')]/following-sibling::*[1]",
                                    field_name="Sanctioned Built-up Area")
    Longitude = safe_text(
        driver, '//*[@id="hidden_div"]/div/div/div/div/form/fieldset/div[4]/div[1]/fieldset/div/fieldset/form/div/div[1]/div/project-geo-tagging-preview/fieldset/form/div/fieldset/div/div[1]/div/div',
        field_name="Longitude")
    Latitude = safe_text(
        driver, '//*[@id="hidden_div"]/div/div/div/div/form/fieldset/div[4]/div[1]/fieldset/div/fieldset/form/div/div[1]/div/project-geo-tagging-preview/fieldset/form/div/fieldset/div/div[2]/div/div',
        field_name="Latitude")
    Litigation_Details = safe_text(
        driver, '//*[@id="hidden_div"]/div/div/div/div/form/fieldset/div[4]/div[7]/div/div[2]/div[1]/label[2]',
        field_name="Litigation_Details (header)")
    Address = safe_text(driver, "//*[contains(text(), 'Address')]/following-sibling::*[1]", field_name="Address")
    Street_Name = safe_text(driver, "//*[contains(text(), 'Street Name ')]/following-sibling::*[1]", field_name="Street Name")

    Street_Name1 = safe_text(driver, "//*[contains(text(), 'Locality')]/following-sibling::*[1]",
                              timeout=4, field_name="Locality")

    State_UT = safe_text(driver, "//*[contains(text(), 'State/UT')]/following-sibling::*[1]", field_name="State/UT")
    District = safe_text(driver, "//*[contains(text(), 'District')]/following-sibling::*[1]", field_name="District")
    Taluka = safe_text(driver, "//*[contains(text(), 'Taluka')]/following-sibling::*[1]", field_name="Taluka")
    Village = safe_text(driver, "//*[contains(text(), 'Village')]/following-sibling::*[1]",
                         timeout=4, field_name="Village")

    Pin_Code = safe_text(driver, "//*[contains(text(), 'Pin Code')]/following-sibling::*[1]", field_name="Pin Code")
    Boundaries_East = safe_text(driver, "//*[contains(text(), 'Boundaries East')]/following-sibling::*[1]", field_name="Boundaries East")
    Boundaries_West = safe_text(driver, "//*[contains(text(), 'Boundaries West')]/following-sibling::*[1]", field_name="Boundaries West")
    Boundaries_South = safe_text(driver, "//*[contains(text(), 'Boundaries South')]/following-sibling::*[1]", field_name="Boundaries South")
    Boundaries_North = safe_text(driver, "//*[contains(text(), 'Boundaries North')]/following-sibling::*[1]", field_name="Boundaries North")

    try:
        Promoter_Type = wait.until(
            EC.visibility_of_element_located((By.XPATH, "//*[contains(text(), 'Promoter Type')]/following-sibling::*[1]"))
        ).text
        Name_of_Partnership = wait.until(
            EC.visibility_of_element_located((By.XPATH, "//*[contains(text(), 'Name of Partnership')]/following-sibling::*[1]"))
        ).text
        Promoter_Details = clean_str(f"{Promoter_Type} - {Name_of_Partnership}")
    except Exception:
        Promoter_Details = safe_text(
            driver, '//*[@id="hidden_div"]/div/div/div/div/form/fieldset/div[4]/div[1]/fieldset/div/fieldset/form/div/div[2]/div/div/project-general-information-preview/fieldset/form/div[2]/div',
            field_name="Promoter_Details")

    Are = safe_text(
        driver, '//*[@id="hidden_div"]/div/div/div/div/form/fieldset/div[4]/div[6]/div/div[2]/div[1]/label[2]/b',
        field_name="Are there any Investor other than the Promoter")

    past_experience_info = None
    try:
        promoter_btn_xpath = "//button[normalize-space()='Promoter Past Experience Details']"
        button = WebDriverWait(driver, 8).until(EC.presence_of_element_located((By.XPATH, promoter_btn_xpath)))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
        time.sleep(0.2)
        driver.execute_script("arguments[0].click();", button)
        log("✅ Clicked on 'Promoter Past Experience Details' button.")

        past_div = WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.XPATH, "//button[contains(., 'Promoter Past Experience Details')]/following::div[1]"))
        )
        past_experience_info = clean_str(past_div.find_elements(By.TAG_NAME, "div")[0].find_elements(By.TAG_NAME, "label")[1].text)
        log(f"Past Experience Info: {past_experience_info}")
    except Exception as e:
        log(f"⚠️ Promoter Past Experience Details not available: {e}")

    try:
        litigation_table = WebDriverWait(driver, 8).until(
            EC.visibility_of_element_located((By.XPATH, "//table[contains(., 'Litigation')]"))
        )
        keys = [clean_str(th.text) for th in litigation_table.find_elements(By.TAG_NAME, "th")]
        past_experience_details = {k: [] for k in keys if k}
        for tr in litigation_table.find_elements(By.XPATH, ".//tbody/tr"):
            for count, td in enumerate(tr.find_elements(By.TAG_NAME, "td")):
                if count < len(keys) and keys[count]:
                    past_experience_details[keys[count]].append(clean_str(td.text))
    except TimeoutException:
        past_experience_details = {}
        log("❌ Litigation table not found.")

    summary_bhk = {}
    soup = BeautifulSoup(driver.page_source, "html.parser")
    target = soup.find(string=lambda t: t and "Summary of Apartments/Units" in t)
    if target:
        next_div = target.find_next("div")
        rows = next_div.find_all("tr")
        if rows:
            headers = [clean_str(th.get_text(strip=True)) for th in rows[0].find_all("th")]
            summary_bhk = {h: [] for h in headers if h}
            for row in rows[1:]:
                for h, cell in zip(headers, row.find_all("td")):
                    if h:
                        summary_bhk[h].append(clean_str(cell.get_text(strip=True)))
    else:
        log("⚠️ Summary table not found / empty structure")

    bhk_details = {}
    building_details = {}
    soup = BeautifulSoup(driver.page_source, "html.parser")
    target = soup.find(string=lambda t: t and "Building Details" in t)
    if target:
        next_div = target.find_next("div")
        rows = next_div.find_all("tr")
        if rows:
            headers = [clean_str(th.get_text(strip=True)) for th in rows[0].find_all("th")]
            building_details = {h: [] for h in headers if h}
            view_index = headers.index("View") if "View" in headers else None

            for row in rows[1:]:
                for h, cell in zip(headers, row.find_all("td")):
                    if h:
                        building_details[h].append(clean_str(cell.get_text(strip=True)))

            try:
                div_element = driver.find_element(By.XPATH, "//div[contains(., 'Building Details')]/following-sibling::div[1]")
                selenium_table = div_element.find_element(By.TAG_NAME, "table")
                selenium_rows = selenium_table.find_elements(By.TAG_NAME, "tr")[1:-1]

                for row in selenium_rows:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    if len(cells) > 1:
                        building_name = clean_str(cells[1].text)
                        log(f"Building Name: {building_name}")
                        if view_index is not None and view_index < len(cells):
                            try:
                                view_cell = cells[view_index]
                                btn = view_cell.find_element(By.XPATH, ".//*")
                                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                                time.sleep(0.2)
                                driver.execute_script("arguments[0].click();", btn)

                                modal_locator = (By.CSS_SELECTOR, "div.modal.show")
                                WebDriverWait(driver, 6).until(EC.visibility_of_element_located(modal_locator))

                                modal_soup = BeautifulSoup(driver.page_source, "html.parser")
                                modal = modal_soup.find(class_="modal-dialog modal-dialog-centered custom-modal-dialog")
                                if modal:
                                    modal_table = modal.find_all(class_="col-lg-12")[1]
                                    keys = [clean_str(th.get_text(strip=True)) for th in modal_table.find("thead").find_all("th")]
                                    details = {k: [] for k in keys if k}
                                    for r in modal_table.find("tbody").find_all("tr"):
                                        for k, cell_td in zip(keys, r.find_all("td")):
                                            if k:
                                                details[k].append(clean_str(cell_td.get_text(strip=True)))
                                    bhk_details[building_name] = details
                            except Exception as e:
                                log(f"⚠️ Building modal for {building_name}: {e}")
                            finally:
                                close_modal(driver)
            except Exception as e:
                log(f"⚠️ Building details table: {e}")

    bank_name = ifsc_code = bank_address = None
    try:
        bank_details = driver.find_elements(
            By.XPATH, "//*[contains(text(), 'Bank Details')]/parent::div/following-sibling::div"
        )[0]
        all_details = bank_details.find_elements(By.TAG_NAME, "label")
        bank_name = clean_str(all_details[0].find_element(By.XPATH, "following-sibling::div[1]").text)
        ifsc_code = clean_str(all_details[1].find_element(By.XPATH, "following-sibling::div[1]").text)
        bank_address = clean_str(all_details[2].find_element(By.XPATH, "following-sibling::div[1]").text)
    except Exception as e:
        log(f"⚠️ Bank details not found: {e}")

    all_parking_details = {}
    if "Parking Details" in driver.page_source:
        opened = False
        for _ in range(3):
            try:
                parking_button = driver.find_element(By.XPATH, "//button[.//div[contains(., 'Parking Details')]]")
                driver.execute_script("arguments[0].click();", parking_button)
                time.sleep(2)
                opened = True
                break
            except Exception:
                driver.execute_script("window.scrollBy(0, 150);")
                time.sleep(1)
        if opened:
            tables = driver.find_elements(
                By.XPATH, "//button[.//div[contains(., 'Parking Details')]]/parent::h2/following-sibling::fieldset//table"
            )
            for table in tables:
                try:
                    parent_div = table.find_element(By.XPATH, "./parent::div")
                    grandparent_div = parent_div.find_element(By.XPATH, "./parent::div")
                    building_name_div = grandparent_div.find_element(By.XPATH, "./preceding-sibling::div[1]")
                    building_name = clean_str(building_name_div.text)
                    keys = [clean_str(th.text) for th in table.find_elements(By.TAG_NAME, "th")]
                    parking_details = {k: [] for k in keys if k}
                    for tr in table.find_element(By.TAG_NAME, "tbody").find_elements(By.TAG_NAME, "tr")[:-1]:
                        for idx, td in enumerate(tr.find_elements(By.TAG_NAME, "td")):
                            if idx < len(keys) and keys[idx]:
                                parking_details[keys[idx]].append(clean_str(td.text))
                    all_parking_details[building_name] = parking_details
                except Exception as e:
                    log(f"⚠️ Parking table skipped: {e}")

    project_details = {
        "Registration Number": [Registration_Number],
        "Date of Registration": [Date_of_Registration],
        "Rera ID": [Registration_Number],
        "Project Location": [project_location],
        "Project Type": [project_type],
        "Project Name": [project_name],
        "survey_no": [survey_no],
        "Permissible Builtup Area": [Permissible_Builtup],
        "Sanctioned Builtup Area": [Sanctioned_Builtup],
        "Total Land Area": [Total_Land_Area],
        "Aggregate area": [Aggregate_area_in_sqmt_of_recreational_open_space],
        "Land Area for Project Applied for this Registration": [Land_Area_for_Project_Applied_for_this_Registration],
        "Project proposed completion date original": [project_proposed_completion_date_original],
        "Project proposed completion date revised": [project_proposed_completion_date_revised],
        "Longitude": [Longitude],
        "Latitude": [Latitude],
        "Are there any Investor other than the Promoter in the Project": [Are],
        "State_UT": [State_UT],
        "District": [District],
        "Taluka": [Taluka],
        "Village": [Village],
        "Pin_Code": [Pin_Code],
        "Boundaries_East": [Boundaries_East],
        "Boundaries_West": [Boundaries_West],
        "Boundaries_South": [Boundaries_South],
        "Boundaries_North": [Boundaries_North],
        "Promoter_Details": [Promoter_Details],
        "bhk_details": [bhk_details],
        "building_details": [building_details],
        "Summary_of_Apartments/Units": [summary_bhk],
        "past_experience_info": [past_experience_info],
        "Litigations related to the project ?": [Litigation_Details],
        "All_Parking_Details": [all_parking_details],
        "Bank Name": [bank_name],
        "IFSC Code": [ifsc_code],
        "Bank Address": [bank_address],
        "ProjectStatus": [Project_Status],
        "Locality": [Street_Name1],
    }

    df = pd.DataFrame(project_details)
    return df


In [7]:
# ============================================================
# CELL 6: MULTITHREADED WORKER & PAGE SCRAPER ENGINE
# ============================================================
def scrape_single_project_task(task_info, checkpoint_data):
    page_number = task_info["page_number"]
    idx = task_info["project_index"]
    project_url = task_info["project_url"]
    organization = task_info["organization"]
    last_modified = task_info["last_modified"]
    certificate_dates = task_info["certificate_dates"]

    if is_project_already_scraped(checkpoint_data, project_url):
        log(f"⏩ Project {idx + 1} on page {page_number} ({project_url}) already scraped — skipping.")
        return True, None

    log(f"🚀 Opening project {idx + 1} on page {page_number}: {project_url}")

    for attempt in range(1, MAX_PROJECT_RETRIES + 1):
        driver = get_thread_driver()

        try:
            record_interrupted_project(checkpoint_data, page_number, idx, project_url, reason="in_progress")

            driver.get(project_url)
            driver.execute_script("document.body.style.zoom='75%'")

            try:
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "captcahCanvas")))
            except TimeoutException:
                pass

            captcha_img_file = f"captcha_p{page_number}_i{idx}_t{threading.get_ident()}.png"
            captcha_ok = solve_captcha_fully_automated(driver, captcha_img_file)

            if os.path.exists(captcha_img_file):
                try:
                    os.remove(captcha_img_file)
                except Exception:
                    pass

            if not captcha_ok:
                raise Exception("Captcha solve failed after retries")

            df = extract_project_data(driver)
            df["Organization"] = organization
            df["Last_Modified"] = last_modified
            for k, v in certificate_dates.items():
                df[k] = v

            rera_id = None
            if "Registration Number" in df.columns and len(df["Registration Number"]) > 0:
                rera_id = df["Registration Number"].iloc[0]

            if rera_id and str(rera_id).strip() in load_existing_csv_rera_ids():
                log(f"RERA ID {rera_id} already in {OUTPUT_CSV} -- marking completed.")
                mark_project_completed(checkpoint_data, project_url, rera_id=rera_id)
                return True, None

            with csv_lock:
                saved_df = save_scraped_project_row(df, OUTPUT_CSV, limit=EXCEL_CHAR_LIMIT)
            add_scraped_id_to_cache(rera_id)

            mark_project_completed(checkpoint_data, project_url, rera_id=rera_id)
            log(f"Project {idx + 1} on page {page_number} scraped successfully! (RERA ID: {rera_id})")
            return True, saved_df

        except Exception as e:
            err_msg = str(e)
            log(f"⚠️ Project {idx + 1} on page {page_number} (attempt {attempt}/{MAX_PROJECT_RETRIES}) error: {err_msg}")
            record_interrupted_project(checkpoint_data, page_number, idx, project_url, reason=f"error_{attempt}: {err_msg}")

            if "invalid session id" in err_msg.lower() or "disconnected" in err_msg.lower() or "no such window" in err_msg.lower():
                log("💥 Browser session closed/disconnected — restarting driver for thread...")
                close_thread_driver()

            if attempt < MAX_PROJECT_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(1, 2))

    log(f"❌ Project {idx + 1} on page {page_number} failed permanently after {MAX_PROJECT_RETRIES} attempts.")
    log_failed_project(page_number, idx, project_url, f"Failed after {MAX_PROJECT_RETRIES} retries")
    return False, None


def fetch_page_project_tasks(driver, page_number, checkpoint_data, max_projects_per_page=10, max_retries=MAX_PAGE_RETRIES):
    for attempt in range(1, max_retries + 1):
        if not go_to_results_page(driver, page_number, refresh_on_fail=True):
            log(f"⚠️ Could not load search result page {page_number} (attempt {attempt}/{max_retries})")
            if attempt < max_retries:
                time.sleep(RETRY_BACKOFF_SECONDS)
            continue

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.row.shadow.p-3.mb-5.bg-body.rounded"))
            )
        except TimeoutException:
            if check_and_dismiss_no_records_popup(driver):
                log(f"🔄 'No Records Found' popup appeared on page {page_number}. Refreshing...")
                reset_driver_page_state(driver)
            else:
                log(f"⚠️ Project sections did not appear on page {page_number} within timeout.")
                reset_driver_page_state(driver)
            if attempt < max_retries:
                time.sleep(RETRY_BACKOFF_SECONDS)
            continue

        sections = driver.find_elements(By.CSS_SELECTOR, "div.row.shadow.p-3.mb-5.bg-body.rounded")
        if not sections:
            log(f"⚠️ 0 project sections found on page {page_number} (attempt {attempt}/{max_retries}).")
            reset_driver_page_state(driver)
            if attempt < max_retries:
                time.sleep(RETRY_BACKOFF_SECONDS)
            continue

        # --- Certificate button: try multiple selectors to match the eye-icon button ---
        # Priority 1: <a> with title 'View Certificate' and data-bs-target (original)
        certificate_buttons = driver.find_elements(
            By.XPATH, '//a[@title="View Certificate" and @data-bs-target="#exampleModal"]'
        )
        # Priority 2: any <button> or <a> inside a card whose visible text or title says 'Certificate'
        if not certificate_buttons:
            certificate_buttons = driver.find_elements(
                By.XPATH,
                '//*[self::button or self::a]'
                '[contains(translate(normalize-space(.), "CERTIFICATE", "certificate"), "certificate")'
                ' or contains(translate(@title, "CERTIFICATE", "certificate"), "certificate")'
                ' or @data-bs-target="#exampleModal"]'
            )
        # Priority 3: any element that triggers #exampleModal (catches icon-only buttons)
        if not certificate_buttons:
            certificate_buttons = driver.find_elements(
                By.XPATH, '//*[@data-bs-target="#exampleModal" or @data-target="#exampleModal"]'
            )
        log(f"Found {len(certificate_buttons)} certificate button(s) on page {page_number}")
        project_count = min(len(sections), max_projects_per_page)
        log(f"Found {len(sections)} project sections on page {page_number} (preparing tasks for {project_count})")

        tasks = []
        wait = WebDriverWait(driver, 8)

        for idx in range(project_count):
            section = sections[idx]
            cert_button = certificate_buttons[idx] if idx < len(certificate_buttons) else None

            try:
                try:
                    organization = section.find_element(By.XPATH, ".//p[@class='darkBlue bold ']").text.strip()
                except Exception:
                    organization = None
                try:
                    last_modified = section.find_element(
                        By.XPATH, ".//div[@class='greyColor' and text()='Last Modified']/following-sibling::p"
                    ).text.strip()
                except Exception:
                    last_modified = None

                view_details_link = section.find_element(By.LINK_TEXT, "View Details")
                href_value = view_details_link.get_attribute("href")

                certificate_dates = {}
                already_done = is_project_already_scraped(checkpoint_data, href_value)
                if cert_button is not None and not already_done:
                    certificate_dates = extract_certificate_dates(driver, cert_button, wait)

                tasks.append({
                    "page_number": page_number,
                    "project_index": idx,
                    "project_url": href_value,
                    "organization": organization,
                    "last_modified": last_modified,
                    "certificate_dates": certificate_dates
                })

            except Exception as e:
                log(f"⚠️ Error preparing task for project {idx + 1} on page {page_number}: {e}")

        if tasks:
            return tasks

    log(f"❌ Failed to fetch project tasks for page {page_number} after {max_retries} attempts.")
    return []


In [8]:
# ============================================================
# CELL 7: MAIN MULTITHREADED RUN CONTROLLER
# ============================================================
def run_scrape_multithreaded(page_start, page_end):
    checkpoint_data = load_checkpoint()

    checkpoint_data["job_start_page"] = min(checkpoint_data.get("job_start_page", page_start), page_start)
    save_checkpoint_data(checkpoint_data)
    start_page = checkpoint_data["job_start_page"]

    if start_page != page_start:
        log(f"Resuming from Page {start_page} (checkpoint floor) in sequence.")

    master_driver = create_driver(HEADLESS)
    combined_df = pd.DataFrame()
    consecutive_page_failures = 0

    try:
        for page in range(start_page, page_end):
            completed_pages = checkpoint_data.get("completed_pages", [])
            if page in completed_pages:
                log(f"⏩ Page {page} already completed — skipping.")
                continue

            if not driver_is_alive(master_driver):
                log("💥 Master browser session died — restarting Chrome...")
                try:
                    master_driver.quit()
                except Exception:
                    pass
                master_driver = create_driver(HEADLESS)
                reset_driver_page_state(master_driver)
                consecutive_page_failures = 0

            log(f"➡️ Processing Page {page}/{page_end - 1}...")

            tasks = fetch_page_project_tasks(master_driver, page, checkpoint_data)
            if not tasks:
                consecutive_page_failures += 1
                log_failed_page(page, "Failed to load/parse search results page")

                if consecutive_page_failures >= RESTART_DRIVER_ON_CONSECUTIVE_FAILURES:
                    log(f"🔄 {consecutive_page_failures} consecutive page failures. Restarting master driver...")
                    try:
                        master_driver.quit()
                    except Exception:
                        pass
                    master_driver = create_driver(HEADLESS)
                    reset_driver_page_state(master_driver)
                    consecutive_page_failures = 0

                if not SKIP_FAILED_PAGES:
                    log("Stopping run as SKIP_FAILED_PAGES is False.")
                    break
                continue

            consecutive_page_failures = 0

            active_tasks = [
                t for t in tasks
                if not is_project_already_scraped(checkpoint_data, t["project_url"])
            ]
            skipped_count = len(tasks) - len(active_tasks)
            if skipped_count:
                log(f"⏩ Skipping {skipped_count} already-scraped project(s) on page {page}.")

            if not active_tasks:
                log(f"✅ All projects on page {page} already processed.")
                mark_page_completed(checkpoint_data, page)
                continue

            log(f"⚡ Launching ThreadPoolExecutor with {MAX_WORKERS} workers for {len(active_tasks)} project tasks on page {page}...")

            page_results = []
            all_succeeded = True
            with ThreadPoolExecutor(max_workers=MAX_WORKERS, thread_name_prefix="ScraperWorker") as executor:
                future_to_task = {
                    executor.submit(scrape_single_project_task, task, checkpoint_data): task
                    for task in active_tasks
                }

                for future in as_completed(future_to_task):
                    task = future_to_task[future]
                    try:
                        success, df = future.result()
                        if success and df is not None:
                            page_results.append(df)
                        elif not success:
                            all_succeeded = False
                    except Exception as exc:
                        all_succeeded = False
                        log(f"💥 Worker exception on project {task['project_index'] + 1}: {exc}")

            if page_results:
                page_df = pd.concat(page_results, ignore_index=True)
                combined_df = pd.concat([combined_df, page_df], ignore_index=True)

            still_missing = [
                t for t in tasks
                if not is_project_already_scraped(checkpoint_data, t["project_url"])
            ]
            if not still_missing:
                mark_page_completed(checkpoint_data, page)
                log(f"🏁 Completed processing page {page}.")
            else:
                log(f"⚠️ Page {page} has {len(still_missing)} project(s) still unscraped.")
                if not SKIP_FAILED_PROJECTS:
                    log("Stopping run as SKIP_FAILED_PROJECTS is False.")
                    break

    except (KeyboardInterrupt, SystemExit):
        log("🛑 Manual Interrupt received! Progress recorded in checkpoint.")
    except Exception as exc:
        log(f"💥 Unexpected error in run loop: {exc}")
        log(traceback.format_exc())
    finally:
        log("🧹 Cleaning up master browser session...")
        try:
            master_driver.quit()
        except Exception:
            pass
        close_thread_driver()

    return combined_df


In [ ]:
# ============================================================
# CELL 8: EXECUTE SCRAPER & EXCEL EXPORT
# ============================================================
all_data = run_scrape_multithreaded(START_PAGE, END_PAGE)

if AUTO_RETRY_FAILED_AT_END:
    retry_pages = get_failed_pages()
    if retry_pages:
        log(f"🔁 Auto-retrying {len(retry_pages)} previously failed page(s): {retry_pages}")
        for p in retry_pages:
            retry_df = run_scrape_multithreaded(p, p + 1)
            all_data = pd.concat([all_data, retry_df], ignore_index=True)
    else:   
        log("✅ No failed pages to retry.")

log(f"✅ Run finished. {len(all_data)} new project rows scraped. Check {OUTPUT_CSV}.")

if AUTO_EXPORT_EXCEL:
    log("Starting automated post-scrape Excel export & verification...")
    export_and_verify_dataset(OUTPUT_CSV, OUTPUT_XLSX, limit=EXCEL_CHAR_LIMIT)


[2026-10-12 17:33:08] [MainThread] Granular checkpoint loaded (2293 scraped projects, 237 completed pages, job_start_page=1).
[2026-10-12 17:33:15] [MainThread] ⏩ Page 1 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 2 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 3 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 4 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 5 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 6 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 7 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 8 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 9 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 10 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 11 already completed — skipping.
[2026-10-12 17:33:15] [MainThread] ⏩ Page 12 already compl